In [ ]:
model = "llama3.2:1B"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [ ]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Define the prompt template
summarization_prompt = ChatPromptTemplate([
    ("system", "Summarize the given text in exactly one sentence."),
    ("human", "{text}"),
])

# ADD HERE YOUR CODE
# Create the LLMChain
summarization_chain = summarization_prompt | llm

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# ADD HERE YOUR CODE
# Invoke the chain
summary = summarization_chain.invoke({"text": text})
print(summary.content)

Deep learning has seen significant growth and advancements over the past decade, with large language models being one of the most prominent examples, and is expected to have a major impact on various industries, societies, and individuals.


In [3]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

Deep learning technologies, particularly large language models, are having a significant impact on industries and society, driven by advancements in technology and a wide range of potential applications.

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [4]:
from langchain_core.tools import tool

# ADD HERE YOUR CODE
# Create custom tool

@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiply two integer numbers."""
    return first_int * second_int
  


print(multiply.name)
print(multiply.description)
print(multiply.args) # -> definition of tool arguments

# Invoke custom tool
multiply.invoke({"first_int": 4, "second_int": 5})

multiply
Multiply two integer numbers.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}


20

In [5]:
# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': 5, 'second_int': 42}, 'id': '3cf7dc55-37c6-463f-86fd-575d9a0df2ef', 'type': 'tool_call'}]


In [6]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
chain_with_tools = llm_with_tools | (lambda msg: multiply.invoke(msg.tool_calls[0]["args"]))

# Run chain
chain_with_tools.invoke("whats 5 times forty two")

210

#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define the system prompt.
- Define tools.
- Create an Agent using the Ollama model, system-prompt and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To](https://docs.langchain.com/oss/javascript/langchain/agents)

In [16]:
from langchain.agents import create_agent

# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
agent_prompt = (
    "You are a math assistant. "
    "Use only the tools add and exponentiate. "
    "For powers, use exponentiate(base, exponent). "
    "For addition, use add(first_int, second_int). "
    "For example, to calculate 2 to the power of 3 and then add 4, "
    "first call exponentiate with base=2 and exponent=3, "
    "then call add with first_int=8 and second_int=4. "
    "Always complete both steps before giving the final answer."
)


# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent


tools = [add, exponentiate]

In [17]:
# ADD HERE YOUR CODE
# Construct the agent
agent_with_tools = create_agent(model=llm, tools=tools, system_prompt=agent_prompt)

In [ ]:
response = agent_with_tools.invoke(
    {"messages": [{"role": "user", "content": "First take 3 to the power of five and afterwards add 12?"}]}
)

for message in response["messages"]:
    print(message.content)

First take 3 to the power of five and afterwards add 12?
}
{"type":"function","function":{"name":"exponentiate","description":"Exponentiate the base to the exponent power.","parameters":{"type":"object","required":["base","exponent"],"properties":{"base":{"type":"integer"},"exponent":{"type":"integer"}}}}}
{"type":"function","function":{"name":"add","description":"Add two integers.","parameters":{"type":"object","required":["first_int","second_int"],"properties":{"first_int":{"type":"integer"},"second_int":{"type":"integer"}}}}}
{"type":"function","function":{"name":"add","description":"Add two integers.","parameters":{"type":"object","required":["first_int","second_int"],"properties":{"first_int":{"type":"integer"},"second_int":{"type":"integer"}}}}}"
Error invoking tool 'exponentiate' with kwargs {'type': 'object', 'required': ['base', 'exponent'], 'properties': {'base': {'type': 'integer'}, 'exponent': {'type': 'integer'}}} with error:
 base: Field required
exponent: Field required
